In [2]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import PIL
import torch
import torch.nn as nn
import torch.optim as optim
import torchinfo
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix, classification_report
from tqdm import tqdm
from torch.utils.data import DataLoader, Dataset
from torchvision import datasets, transforms
from torchvision.transforms import functional as F
import cv2 # OpenCV for some image processing functions
from skimage import restoration, exposure # Scikit-image for deconvolution, exposure
from PIL import Image, ImageEnhance


import torchvision.models as models


import mlflow
import mlflow.pytorch


c:\Users\tsinamai\AppData\Local\miniconda3\envs\ml\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
class CustomTestDataset(Dataset):
    """
    A custom PyTorch Dataset for loading images from a flat directory structure,
    typically used for unlabelled test data in competitions.
    """
    def __init__(self, root_dir, transform=None):
        """
        Args:
            root_dir (string): Directory with all the test images.
            transform (callable, optional): Optional transform to be applied on a sample.
        """
        self.root_dir = root_dir
        # Filter for common image file extensions
        self.image_files = sorted([f for f in os.listdir(root_dir) if f.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp', '.tiff'))])
        self.transform = transform
        print(f"Found {len(self.image_files)} test images in {root_dir}")

    def __len__(self):
        return len(self.image_files)

    def __getitem__(self, idx):
        img_name = os.path.join(self.root_dir, self.image_files[idx])
        image = PIL.Image.open(img_name).convert("RGB") # Ensure RGB conversion

        if self.transform:
            image = self.transform(image)

        # For test data, we don't have labels. Return a dummy label (e.g., 0)
        # The predict function will ignore this label.
        return image, 0 # Dummy label


In [4]:
def setup_device():
    """
    Automatically detect and set up the best available device for training.

    Returns:
        str: Device string ('cuda', 'mps', or 'cpu')
    """
    if torch.cuda.is_available():
        device = "cuda"
        print(f"Using CUDA GPU: {torch.cuda.get_device_name()}")
    elif torch.backends.mps.is_available():
        device = "mps"
        print("Using Apple Metal Performance Shaders (MPS)")
    else:
        device = "cpu"
        print("Using CPU")
    return device

def setup_data_directories(base_dir="crop pictures"):
    """
    Set up and validate data directory paths.

    Args:
        base_dir (str): Base directory containing train/val/test folders.

    Returns:
        dict: Dictionary containing paths to train, val, and test directories.
    """
    directories = {
        'train': os.path.join(base_dir, "train"),
        'val': os.path.join(base_dir, "val"),
        'test': os.path.join(base_dir, "test") # 'test' here refers to the unlabelled submission data
    }

    # Validate directories exist
    for split, path in directories.items():
        if os.path.exists(path):
            print(f"{split.capitalize()} Data Directory: {path} ✓")
        else:
            print(f"Warning: {split.capitalize()} directory not found: {path}")

    return directories

def get_class_information(train_dir):
    """
    Extract class names and number of classes from the training directory.

    Args:
        train_dir (str): Path to training directory.

    Returns:
        tuple: (classes list, number of classes).
    """
    if not os.path.exists(train_dir):
        raise FileNotFoundError(f"Training directory not found: {train_dir}")

    # Ensure we only list actual directories (classes)
    classes = sorted([d for d in os.listdir(train_dir) if os.path.isdir(os.path.join(train_dir, d))])

    print(f"Found {len(classes)} classes: {classes}")
    return classes, len(classes)

In [5]:
def class_counts(dataset):
    """
    Counts the number of samples per class in a labelled dataset (ImageFolder).

    Args:
        dataset: A PyTorch Dataset object (expected to be torchvision.datasets.ImageFolder).

    Returns:
        A pandas Series with class names as index and counts as values.
    """
    counts = {}
    if hasattr(dataset, 'targets') and hasattr(dataset, 'classes'):
        for label_idx in dataset.targets:
            class_name = dataset.classes[label_idx]
            counts[class_name] = counts.get(class_name, 0) + 1
    elif isinstance(dataset, CustomTestDataset):
        # CustomTestDataset is unlabelled, so we just count total images.
        return pd.Series({'unlabelled': len(dataset)})
    else:
        print("Warning: Dataset does not have 'targets' or 'classes' attribute directly. "
              "Falling back to iterating through dataset, which can be slow.")
        # This fallback iterates through the dataset to get labels
        # It assumes dataset[i][1] gives the label
        for i in tqdm(range(len(dataset)), desc="Counting classes"):
            _, label = dataset[i]
            class_name = getattr(dataset, 'classes', {}).get(label, f"Class_{label}") # Try to get class name
            counts[class_name] = counts.get(class_name, 0) + 1
    return pd.Series(counts)

def analyze_class_distribution(dataset, dataset_name="Dataset"):
    """
    Analyze and print class distribution for a dataset.

    Args:
        dataset: PyTorch Dataset object.
        dataset_name (str): Name of the dataset for display.

    Returns:
        pd.Series: Class distribution as pandas Series.
    """
    print(f"\nAnalyzing {dataset_name}...")
    distribution = class_counts(dataset)
    total_samples = distribution.sum()

    print(f"{dataset_name} class distribution:")
    if 'unlabelled' in distribution.index and len(distribution) == 1:
        print(f"  {total_samples} unlabelled samples.")
    else:
        for class_name, count in distribution.items():
            percentage = (count / total_samples) * 100 if total_samples > 0 else 0.0
            print(f"  {class_name}: {count} ({percentage:.1f}%)")

    return distribution

def detect_class_imbalance(distribution, threshold=0.5):
    """
    Detect class imbalance in the dataset distribution.

    Args:
        distribution (pd.Series): Class distribution (should not be from unlabelled test set).
        threshold (float): Imbalance threshold (minority/majority ratio).

    Returns:
        dict: Imbalance analysis results.
    """
    if 'unlabelled' in distribution.index and len(distribution) == 1:
        print("Skipping imbalance detection for unlabelled dataset.")
        return {
            'is_imbalanced': False, 'imbalance_ratio': 1.0,
            'majority_class': None, 'minority_class': None,
            'majority_count': 0, 'minority_count': 0,
            'total_samples': distribution['unlabelled']
        }

    total_samples = distribution.sum()
    if total_samples == 0:
        print("No samples in dataset for imbalance detection.")
        return {
            'is_imbalanced': False,
            'imbalance_ratio': 1.0,
            'majority_class': None,
            'minority_class': None,
            'majority_count': 0,
            'minority_count': 0,
            'total_samples': 0
        }

    max_count = distribution.max()
    min_count = distribution.min()

    imbalance_ratio = min_count / max_count
    is_imbalanced = imbalance_ratio < threshold

    majority_class = distribution.idxmax()
    minority_class = distribution.idxmin()

    results = {
        'is_imbalanced': is_imbalanced,
        'imbalance_ratio': imbalance_ratio,
        'majority_class': majority_class,
        'minority_class': minority_class,
        'majority_count': max_count,
        'minority_count': min_count,
        'total_samples': total_samples
    }

    print(f"\nClass Imbalance Analysis (Threshold: {threshold}):")
    print(f"  Imbalanced: {'Yes' if is_imbalanced else 'No'}")
    print(f"  Imbalance ratio (Min/Max): {imbalance_ratio:.3f}")
    if is_imbalanced:
        print(f"  Majority class: {majority_class} ({max_count} samples)")
        print(f"  Minority class: {minority_class} ({min_count} samples)")
        print(f"  Recommendation: Consider data augmentation or resampling strategies.")

    return results

In [6]:
class ConvertToRGB:
    """Callable class to convert PIL image to RGB format."""
    def __call__(self, img):
        return img.convert("RGB") if img.mode != "RGB" else img

class ApplyDeblurring:
    """Callable class to apply deblurring techniques."""
    def __init__(self, method="none", psf_size=5):
        self.method = method
        self.psf_size = psf_size

    def __call__(self, image_pil):
        if self.method == "none":
            return image_pil

        image_np = np.array(image_pil)

        if self.method in ["wiener", "lucy_richardson"]:
            psf = np.ones((self.psf_size, self.psf_size)) / (self.psf_size * self.psf_size)

            deblurred_channels = []
            for i in range(image_np.shape[2]):
                channel = image_np[:, :, i]
                if self.method == "wiener":
                    deblurred_channel = restoration.wiener(channel, psf, balance=0.01)
                elif self.method == "lucy_richardson":
                    deblurred_channel = restoration.richardson_lucy(channel, psf, num_iter=20)
                deblurred_channels.append(np.clip(deblurred_channel, 0, 255).astype(np.uint8))

            deblurred_image_np = np.stack(deblurred_channels, axis=-1)
            return Image.fromarray(deblurred_image_np)

        elif self.method == "deep_learning":
            print("Warning: Deep learning deblurring requires a separate pre-trained model. Skipping for now.")
            return image_pil
        else:
            print(f"Unknown deblurring method: {self.method}. Returning original image.")
            return image_pil

class ApplyNoiseReduction:
    """Callable class to apply noise reduction."""
    def __init__(self, method="none", kernel_size=3):
        self.method = method
        self.kernel_size = kernel_size

    def __call__(self, image_pil):
        if self.method == "none":
            return image_pil

        image_np = np.array(image_pil)
        denoised_image_np = image_np

        if self.method == "gaussian":
            denoised_image_np = cv2.GaussianBlur(image_np, (self.kernel_size, self.kernel_size), 0)
        elif self.method == "median":
            # Ensure odd kernel size for median filter
            k_size = self.kernel_size if self.kernel_size % 2 != 0 else self.kernel_size + 1
            denoised_image_np = cv2.medianBlur(image_np, k_size)
        elif self.method == "bilateral":
            # Default parameters for bilateral filter, adjust as needed
            denoised_image_np = cv2.bilateralFilter(image_np, 9, 75, 75)
        else:
            print(f"Unknown noise reduction method: {self.method}. Returning original image.")

        return Image.fromarray(denoised_image_np)

class ApplySharpening:
    """Callable class to apply sharpening."""
    def __init__(self, factor=1.0):
        self.factor = factor

    def __call__(self, image_pil):
        if self.factor == 1.0:
            return image_pil
        enhancer = ImageEnhance.Sharpness(image_pil)
        return enhancer.enhance(self.factor)

class ApplySharpeningRandom:
    """Callable class to apply sharpening with a random factor from a range."""
    def __init__(self, factor_range=(1.0, 1.2)):
        self.factor_range = factor_range

    def __call__(self, image_pil):
        factor = np.random.uniform(self.factor_range[0], self.factor_range[1])
        if factor == 1.0: # Optimization: no change if factor is 1.0
            return image_pil
        enhancer = ImageEnhance.Sharpness(image_pil)
        return enhancer.enhance(factor)


class ApplyContrastEnhancement:
    """Callable class to apply contrast enhancement."""
    def __init__(self, method="none", factor=1.0):
        self.method = method
        self.factor = factor

    def __call__(self, image_pil):
        if self.method == "none":
            return image_pil

        if self.method == "brightness":
            enhancer = ImageEnhance.Brightness(image_pil)
            return enhancer.enhance(self.factor)
        elif self.method == "contrast":
            enhancer = ImageEnhance.Contrast(image_pil)
            return enhancer.enhance(self.factor)
        elif self.method == "equalize":
            return F.equalize(image_pil)
        elif self.method == "adapt_equalize":
            image_np = np.array(image_pil)
            if image_np.ndim == 3: # Color image
                enhanced_channels = []
                for i in range(image_np.shape[2]):
                    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
                    enhanced_channel = clahe.apply(image_np[:, :, i])
                    enhanced_channels.append(enhanced_channel)
                enhanced_image_np = np.stack(enhanced_channels, axis=-1)
            else: # Grayscale
                clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
                enhanced_image_np = clahe.apply(image_np)
            return Image.fromarray(enhanced_image_np)
        else:
            print(f"Unknown contrast enhancement method: {self.method}. Returning original image.")
            return image_pil

class ApplyColorCorrection:
    """Callable class to apply color correction."""
    def __init__(self, method="none", factor=1.0):
        self.method = method
        self.factor = factor

    def __call__(self, image_pil):
        if self.method == "none":
            return image_pil

        if self.method == "saturation":
            enhancer = ImageEnhance.Color(image_pil)
            return enhancer.enhance(self.factor)
        else:
            print(f"Unknown color correction method: {self.method}. Returning original image.")
            return image_pil


# --- Modified create_base_transforms function ---

def create_base_transforms(image_size=(224, 224),
                           deblur_method="none", psf_size=5,
                           noise_reduct_method="none", nr_kernel_size=3,
                           sharpen_factor=1.0,
                           contrast_enhance_method="none", ce_factor=1.0,
                           color_correct_method="none", cc_factor=1.0):
    """
    Create basic image transforms for preprocessing (validation/test).
    Can include deblurring and enhancement if inherent to dataset.
    """
    transform_list = []

    # Always convert to RGB first
    transform_list.append(ConvertToRGB())

    # Add deblurring/enhancement *before* resize
    if deblur_method != "none":
        transform_list.append(ApplyDeblurring(deblur_method, psf_size))
    if noise_reduct_method != "none":
        transform_list.append(ApplyNoiseReduction(noise_reduct_method, nr_kernel_size))
    if contrast_enhance_method != "none":
        transform_list.append(ApplyContrastEnhancement(contrast_enhance_method, ce_factor))
    if color_correct_method != "none":
        transform_list.append(ApplyColorCorrection(color_correct_method, cc_factor))
    if sharpen_factor != 1.0: # Only add if actual sharpening is applied
        transform_list.append(ApplySharpening(sharpen_factor))

    transform_list.append(transforms.Resize(image_size))
    transform_list.append(transforms.ToTensor())
    transform_list.append(transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]))

    transform = transforms.Compose(transform_list)
    print(f"Created base transforms for image size: {image_size}. Deblurring: {deblur_method}, Noise Reduction: {noise_reduct_method}")
    return transform


# --- Modified create_augmented_transforms function ---

def create_augmented_transforms(image_size=(224, 224), augmentation_strength="medium",
                                deblur_method="none", psf_size=5,
                                noise_reduct_method="none", nr_kernel_size=3,
                                sharpen_factor_range=(1.0, 1.2), # For augmentation, use a range
                                contrast_enhance_method="none", ce_factor_range=(1.0, 1.2),
                                color_correct_method="none", cc_factor_range=(1.0, 1.2)):
    """
    Create augmented transforms for training data, including deblurring and enhancement.
    """
    base_transforms = []

    # Always convert to RGB first
    base_transforms.append(ConvertToRGB())

    # Add deblurring/enhancement *before* resize for best effect
    if deblur_method != "none":
        base_transforms.append(ApplyDeblurring(deblur_method, psf_size))
    if noise_reduct_method != "none":
        base_transforms.append(ApplyNoiseReduction(noise_reduct_method, nr_kernel_size))

    # Apply random enhancements for augmentation
    if contrast_enhance_method == "brightness":
        base_transforms.append(transforms.RandomApply([transforms.ColorJitter(brightness=ce_factor_range)], p=0.5))
    elif contrast_enhance_method == "contrast":
        base_transforms.append(transforms.RandomApply([transforms.ColorJitter(contrast=ce_factor_range)], p=0.5))
    elif contrast_enhance_method == "equalize":
        # Using a direct callable class for equalize
        base_transforms.append(transforms.RandomApply([ApplyContrastEnhancement(method="equalize")], p=0.2))
    elif contrast_enhance_method == "adapt_equalize":
        # Using a direct callable class for adapt_equalize
        base_transforms.append(transforms.RandomApply([ApplyContrastEnhancement(method="adapt_equalize")], p=0.2))

    if color_correct_method == "saturation":
        base_transforms.append(transforms.RandomApply([transforms.ColorJitter(saturation=cc_factor_range)], p=0.5))

    if sharpen_factor_range != (1.0, 1.0):
        # Use the custom callable class that handles random factor selection
        base_transforms.append(transforms.RandomApply([
            ApplySharpeningRandom(sharpen_factor_range)
        ], p=0.5))

    base_transforms.append(transforms.Resize(image_size))

    augmentations = []
    if augmentation_strength == "light":
        augmentations = [
            transforms.RandomHorizontalFlip(p=0.3),
            transforms.RandomRotation(degrees=10),
            transforms.ColorJitter(brightness=0.1, contrast=0.1),
        ]
    elif augmentation_strength == "medium":
        augmentations = [
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.RandomVerticalFlip(p=0.2),
            transforms.RandomRotation(degrees=15),
            transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1),
            transforms.RandomAffine(degrees=0, translate=(0.05, 0.05)),
        ]
    elif augmentation_strength == "heavy":
        augmentations = [
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.RandomVerticalFlip(p=0.3),
            transforms.RandomRotation(degrees=20),
            transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2, hue=0.1),
            transforms.RandomAffine(degrees=0, translate=(0.1, 0.1), scale=(0.9, 1.1)),
            transforms.RandomPerspective(distortion_scale=0.2, p=0.3),
        ]
    else:
        raise ValueError("augmentation_strength must be 'light', 'medium', or 'heavy'")

    final_transforms = [
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ]

    transform = transforms.Compose(base_transforms + augmentations + final_transforms)
    print(f"Created {augmentation_strength} augmented transforms (with enhancements).")
    return transform

In [7]:
def create_datasets(data_dirs, train_transform, val_transform, test_transform_for_submission=None):
    """
    Create PyTorch datasets for training, validation, and test submission.

    Args:
        data_dirs (dict): Dictionary with directory paths for 'train', 'val', 'test'.
        train_transform (transforms.Compose): Transform for training data.
        val_transform (transforms.Compose): Transform for validation data.
        test_transform_for_submission (transforms.Compose, optional): Transform for test submission data.
                                                                        If None, val_transform is used.

    Returns:
        dict: Dictionary containing datasets ('train', 'val', 'test_submission').
    """
    datasets_dict = {}

    # Training dataset (expects class subfolders)
    if 'train' in data_dirs and os.path.exists(data_dirs['train']):
        datasets_dict['train'] = datasets.ImageFolder(data_dirs['train'], train_transform)
        print(f'Training dataset created: {len(datasets_dict["train"])} samples')
    else:
        print(f"Warning: Training directory not found at {data_dirs.get('train')}. Skipping train dataset creation.")

    # Validation dataset (expects class subfolders)
    if 'val' in data_dirs and os.path.exists(data_dirs['val']):
        datasets_dict['val'] = datasets.ImageFolder(data_dirs['val'], val_transform)
        print(f'Validation dataset created: {len(datasets_dict["val"])} samples')
    else:
        print(f"Warning: Validation directory not found at {data_dirs.get('val')}. Skipping validation dataset creation.")

    # Test dataset for submission (uses CustomTestDataset for flat structure)
    if 'test' in data_dirs and os.path.exists(data_dirs['test']):
        transform_to_use = test_transform_for_submission if test_transform_for_submission else val_transform
        datasets_dict['test_submission'] = CustomTestDataset(data_dirs['test'], transform_to_use)
        print(f'Test submission dataset created: {len(datasets_dict["test_submission"])} samples')
    else:
        print(f"Warning: Test submission directory not found at {data_dirs.get('test')}. Skipping test submission dataset creation.")

    return datasets_dict

def create_dataloaders(datasets_dict, batch_size=32, num_workers=0):
    """
    Create DataLoaders for all available datasets.

    Args:
        datasets_dict (dict): Dictionary containing datasets.
        batch_size (int): Batch size for data loading.
        num_workers (int): Number of worker processes for data loading.

    Returns:
        dict: Dictionary containing DataLoaders.
    """
    dataloaders = {}
    # Determine if pin_memory should be used
    pin_memory = torch.cuda.is_available()

    for split, dataset in datasets_dict.items():
        # Shuffle only training data
        shuffle = (split == 'train')
        dataloaders[split] = DataLoader(
            dataset,
            batch_size=batch_size,
            shuffle=shuffle,
            num_workers=num_workers,
            pin_memory=pin_memory
        )
        print(f"{split.capitalize()} DataLoader created with {len(dataloaders[split])} batches.")
    return dataloaders

In [8]:
class SimpleCNN(nn.Module):
    """
    A simple CNN model for crop disease classification.
    Designed with a clear feature extractor and a classifier head.
    """
    def __init__(self, num_classes=4, image_size=(224, 224)):
        super().__init__()
        # Feature extractor
        self.features = nn.Sequential(
            nn.Conv2d(3, 8, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Conv2d(8, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2)
        )

        # Calculate the flattened size dynamically to avoid manual errors
        with torch.no_grad():
            dummy_input = torch.zeros(1, 3, image_size[0], image_size[1])
            # Pass through features and flatten
            flattened_size = self.features(dummy_input).view(1, -1).size(1)

        # Classifier head
        self.classifier = nn.Linear(flattened_size, num_classes)

    def forward(self, x):
        features = self.features(x)
        features = features.view(features.size(0), -1) # Flatten
        output = self.classifier(features)
        return output

    # Method to get features (useful for advanced techniques like feature-level augmentation)
    def get_features(self, x):
        return self.features(x).view(x.size(0), -1)

def setup_training_components(num_classes, learning_rate=0.001, model_name="SimpleCNN", image_size=(224, 224), pretrained=True):
    """
    Setup model, optimizer, and loss function.
    Incorporates different models based on model_name.

    Args:
        num_classes (int): Number of output classes.
        learning_rate (float): Learning rate for the optimizer.
        model_name (str): Name of the model to use ('SimpleCNN', 'VGG16', 'ResNet50', 'InceptionV3').
        image_size (tuple): Input image size for model initialization.
        pretrained (bool): Whether to use pre-trained weights (for VGG, ResNet, Inception).

    Returns:
        tuple: (model, optimizer, loss_fn).
    """
    if model_name == "SimpleCNN":
        model = SimpleCNN(num_classes, image_size)
    elif model_name == "VGG16":
        model = models.vgg16(weights=models.VGG16_Weights.IMAGENET1K_V1 if pretrained else None)
        # Modify classifier for our number of classes
        model.classifier[6] = nn.Linear(model.classifier[6].in_features, num_classes)
        print(f"Initialized VGG16 model with {'pretrained' if pretrained else 'random'} weights.")
    elif model_name == "ResNet50":
        model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1 if pretrained else None)
        # Modify the final fully connected layer
        model.fc = nn.Linear(model.fc.in_features, num_classes)
        print(f"Initialized ResNet50 model with {'pretrained' if pretrained else 'random'} weights.")
    elif model_name == "InceptionV3":
        # InceptionV3 requires input size of (299, 299) and expects specific transform behavior
        # Ensure your transforms handle this. Also, it has an auxiliary output in training.
        model = models.inception_v3(weights=models.Inception_V3_Weights.IMAGENET1K_V1 if pretrained else None)
        # Modify the primary output layer
        model.fc = nn.Linear(model.fc.in_features, num_classes)
        # Modify auxiliary output layer if training is involved
        if model.aux_logits:
            model.AuxLogits.fc = nn.Linear(model.AuxLogits.fc.in_features, num_classes)
        print(f"Initialized InceptionV3 model with {'pretrained' if pretrained else 'random'} weights.")
        if image_size != (299, 299):
            print(f"Warning: InceptionV3 typically expects input size (299, 299). Current: {image_size}")
    else:
        raise ValueError(f"Unknown model name: {model_name}. Choose from 'SimpleCNN', 'VGG16', 'ResNet50', 'InceptionV3'.")

    print(f"Model architecture ({model_name}):")
    # For complex models, summary might take time or be too verbose, just print the model
    # torchinfo.summary(model, input_size=(1, 3, image_size[0], image_size[1]), col_names=["input_size", "output_size", "num_params", "kernel_size", "mult_adds"],)
    print(model)

    loss_fn = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=learning_rate)

    print(f"\nTraining Configuration:")
    print(f"Loss Function: CrossEntropyLoss")
    print(f"Optimizer: Adam (Learning Rate: {learning_rate})")

    return model, optimizer, loss_fn

In [9]:
def train_model(
    model,
    optimizer,
    loss_fn,
    train_loader,
    val_loader,
    epochs=5,
    device='cpu',
    model_name="unknown" # Added for MLflow logging
):
    """
    Train a PyTorch model and validate its performance.

    Args:
        model: The neural network model to train.
        optimizer: Optimization algorithm (e.g., Adam).
        loss_fn: Loss function (e.g., CrossEntropyLoss).
        train_loader: DataLoader for training data.
        val_loader: DataLoader for validation data.
        epochs: Number of training iterations.
        device: Device to train on ('cpu', 'cuda', or 'mps').
        model_name (str): Name of the model for logging.

    Returns:
        Tuple of lists containing training/validation losses and accuracies over epochs.
    """
    model.to(device)

    train_losses = []
    val_losses = []
    train_accuracies = []
    val_accuracies = []

    print("\n--- Starting Model Training ---")
    for epoch in range(epochs):
        # --- TRAINING PHASE ---
        model.train()
        running_train_loss = 0.0
        correct_train = 0
        total_train = 0

        for inputs, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs} Training"):
            inputs, labels = inputs.to(device), labels.to(device)

            optimizer.zero_grad()
            
            # Handle InceptionV3's auxiliary output during training
            if model_name == "InceptionV3" and model.training:
                outputs, aux_outputs = model(inputs)
                loss1 = loss_fn(outputs, labels)
                loss2 = loss_fn(aux_outputs, labels)
                loss = loss1 + 0.4 * loss2 # Standard Inception loss weighting
            else:
                outputs = model(inputs)
                loss = loss_fn(outputs, labels)
            
            loss.backward()
            optimizer.step()

            running_train_loss += loss.item() * inputs.size(0) # Multiply by batch size for true sum
            _, predicted = torch.max(outputs.data, 1)
            total_train += labels.size(0)
            correct_train += (predicted == labels).sum().item()

        epoch_train_loss = running_train_loss / total_train
        epoch_train_accuracy = correct_train / total_train
        train_losses.append(epoch_train_loss)
        train_accuracies.append(epoch_train_accuracy)

        # --- VALIDATION PHASE ---
        model.eval()
        running_val_loss = 0.0
        correct_val = 0
        total_val = 0
        all_val_labels = []
        all_val_predictions = []

        with torch.no_grad():
            for inputs, labels in tqdm(val_loader, desc=f"Epoch {epoch+1}/{epochs} Validation"):
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                loss = loss_fn(outputs, labels)

                running_val_loss += loss.item() * inputs.size(0)
                _, predicted = torch.max(outputs.data, 1)
                total_val += labels.size(0)
                correct_val += (predicted == labels).sum().item()

                all_val_labels.extend(labels.cpu().numpy())
                all_val_predictions.extend(predicted.cpu().numpy())

        epoch_val_loss = running_val_loss / total_val
        epoch_val_accuracy = correct_val / total_val
        val_losses.append(epoch_val_loss)
        val_accuracies.append(epoch_val_accuracy)

        print(f"Epoch {epoch+1}/{epochs}: "
              f"Train Loss: {epoch_train_loss:.4f}, Train Acc: {epoch_train_accuracy:.4f} | "
              f"Val Loss: {epoch_val_loss:.4f}, Val Acc: {epoch_val_accuracy:.4f}")
        
        # --- MLflow Logging per epoch ---
        mlflow.log_metric("train_loss", epoch_train_loss, step=epoch)
        mlflow.log_metric("train_accuracy", epoch_train_accuracy, step=epoch)
        mlflow.log_metric("val_loss", epoch_val_loss, step=epoch)
        mlflow.log_metric("val_accuracy", epoch_val_accuracy, step=epoch)
        # --- End MLflow Logging per epoch ---


    print("\n--- Training Complete ---")
    
    # --- MLflow Log Final Metrics ---
    # Log the best validation accuracy and corresponding loss
    best_val_accuracy = max(val_accuracies)
    best_val_epoch = val_accuracies.index(best_val_accuracy)
    best_val_loss = val_losses[best_val_epoch]
    
    mlflow.log_metric("final_train_loss", train_losses[-1])
    mlflow.log_metric("final_train_accuracy", train_accuracies[-1])
    mlflow.log_metric("final_val_loss", best_val_loss)
    mlflow.log_metric("final_val_accuracy", best_val_accuracy)

    # Log classification report
    report = classification_report(all_val_labels, all_val_predictions, target_names=val_loader.dataset.classes, output_dict=True)
    for class_name, metrics in report.items():
        if isinstance(metrics, dict):
            for metric_name, value in metrics.items():
                if metric_name in ['precision', 'recall', 'f1-score', 'support']:
                    mlflow.log_metric(f"val_{class_name}_{metric_name}", value)
        elif class_name in ['accuracy', 'macro avg', 'weighted avg']: # Handle overall metrics
            if isinstance(metrics, float): # For accuracy, which is a float directly
                mlflow.log_metric(f"val_{class_name}", metrics)
            elif isinstance(metrics, dict): # For macro avg, weighted avg
                 for metric_name, value in metrics.items():
                    if metric_name in ['precision', 'recall', 'f1-score', 'support']:
                        mlflow.log_metric(f"val_{class_name.replace(' ', '_')}_{metric_name}", value)


    # Log Confusion Matrix as an artifact
    cm = confusion_matrix(all_val_labels, all_val_predictions)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=val_loader.dataset.classes)
    disp.plot(cmap=plt.cm.Blues)
    plt.title("Validation Confusion Matrix")
    plt.savefig("confusion_matrix.png")
    mlflow.log_artifact("confusion_matrix.png")
    plt.close() # Close plot to free memory

    # Save the final model
    mlflow.pytorch.log_model(model, "model")
    # --- End MLflow Log Final Metrics ---

    return train_losses, val_losses, train_accuracies, val_accuracies

def predict_on_test_data(model, test_loader, device, classes):
    """
    Generate predictions for the test dataset.

    Args:
        model: The trained neural network model.
        test_loader: DataLoader for test data (CustomTestDataset).
        device: Device to run inference on.
        classes: List of class names.

    Returns:
        pd.DataFrame: DataFrame with image filenames and predicted labels.
    """
    model.eval()
    model.to(device)
    predictions = []
    image_filenames = test_loader.dataset.image_files # Access filenames from CustomTestDataset

    print("\n--- Generating Test Predictions ---")
    with torch.no_grad():
        for i, (inputs, _) in enumerate(tqdm(test_loader, desc="Predicting on Test Set")):
            inputs = inputs.to(device)
            outputs = model(inputs)
            _, predicted = torch.max(outputs.data, 1)
            predictions.extend(predicted.cpu().numpy())

    # Map numeric predictions back to class names
    predicted_class_names = [classes[p] for p in predictions]

    results_df = pd.DataFrame({
        'Image': image_filenames,
        'Label': predicted_class_names
    })
    return results_df

def save_submission_file(predictions_df, filename="submission.csv"):
    """
    Save the predictions DataFrame to a CSV file.

    Args:
        predictions_df (pd.DataFrame): DataFrame with 'Image' and 'Label' columns.
        filename (str): Name of the CSV file to save.
    """
    predictions_df.to_csv(filename, index=False)
    print(f"\nSubmission file saved to {filename}")


In [10]:
if __name__ == "__main__":
    # --- Basic Setup ---
    device = setup_device()
    data_dirs = setup_data_directories()
    
    # Get class information from training directory
    train_dir_path = data_dirs.get('train')
    if train_dir_path:
        classes, num_classes = get_class_information(train_dir_path)
    else:
        print("Cannot get class information without a valid training directory.")
        classes = []
        num_classes = 0 # Default to 0, or handle error appropriately

    image_size = (224, 224) # Default for VGG/ResNet, Inception needs (299, 299)
    batch_size = 32
    num_epochs = 10 # Increase epochs for better training
    num_workers = 0 # Adjust based on your system's CPU cores for faster data loading

    # Define a dictionary of models to experiment with
    # For InceptionV3, ensure image_size is (299, 299) and adapt transforms if necessary
    model_configs = {
        "VGG16": {"image_size": (224, 224), "pretrained": True},
        "ResNet50": {"image_size": (224, 224), "pretrained": True},
        "InceptionV3": {"image_size": (299, 299), "pretrained": True}, # InceptionV3 specific size
        # "SimpleCNN": {"image_size": (224, 224), "pretrained": False} # Add your custom CNN
    }

    # --- MLflow Experiment Setup ---
    mlflow.set_tracking_uri("http://localhost:5000") # Or your remote MLflow tracking server
    mlflow.set_experiment("Image_Detection_Competition_V2") # Name your experiment

    for model_name, config in model_configs.items():
        with mlflow.start_run(run_name=f"{model_name}_Pretrained"):
            # --- Log Model Parameters to MLflow ---
            mlflow.log_param("model_architecture", model_name)
            mlflow.log_param("image_size", config["image_size"])
            mlflow.log_param("pretrained", config["pretrained"])
            mlflow.log_param("batch_size", batch_size)
            mlflow.log_param("num_epochs", num_epochs)
            mlflow.log_param("learning_rate", 0.001)
            mlflow.log_param("device", device)
            mlflow.log_param("augmentation_strength", "medium") # Example param
            mlflow.log_param("deblur_method", "none") # Example param
            mlflow.log_param("noise_reduct_method", "none") # Example param

            print(f"\n--- Starting Experiment for Model: {model_name} ---")

            # --- Data Transforms ---
            # Adjust image_size based on model requirements
            current_image_size = config["image_size"]
            
            # Using ImageNet normalization for pre-trained models
            # The create_base_transforms and create_augmented_transforms already include it.
            
            train_transform = create_augmented_transforms(
                image_size=current_image_size, 
                augmentation_strength="medium",
                # Include any desired image enhancements here too
                deblur_method="wiener", psf_size=5,
                noise_reduct_method="bilateral", nr_kernel_size=5,
                sharpen_factor_range=(1.0, 1.2),
                # contrast_enhance_method="adapt_equalize",
                # color_correct_method="saturation", cc_factor_range=(0.8, 1.2)
            )
            val_transform = create_base_transforms(
                image_size=current_image_size,
                deblur_method="wiener", psf_size=5,
                noise_reduct_method="bilateral", nr_kernel_size=5,
                sharpen_factor=1.0,
                # contrast_enhance_method="adapt_equalize",
                # color_correct_method="saturation"
            )
            test_transform_for_submission = val_transform # Usually same as validation

            datasets_dict = create_datasets(data_dirs, train_transform, val_transform, test_transform_for_submission)
            dataloaders = create_dataloaders(datasets_dict, batch_size=batch_size, num_workers=num_workers)

            # --- Model, Optimizer, Loss Setup ---
            model, optimizer, loss_fn = setup_training_components(
                num_classes=num_classes, 
                learning_rate=0.001, 
                model_name=model_name, 
                image_size=current_image_size, 
                pretrained=config["pretrained"]
            )

            # --- Train Model ---
            if 'train' in dataloaders and 'val' in dataloaders:
                train_losses, val_losses, train_accuracies, val_accuracies = train_model(
                    model, optimizer, loss_fn,
                    dataloaders['train'], dataloaders['val'],
                    epochs=num_epochs, device=device, model_name=model_name
                )
            else:
                print("Skipping training due to missing train/val data.")
                continue # Move to next model config

            # --- Make and Save Predictions for Test Data ---
            if 'test_submission' in dataloaders and classes:
                predictions_df = predict_on_test_data(model, dataloaders['test_submission'], device, classes)
                submission_filename = f"submission_{model_name}.csv"
                save_submission_file(predictions_df, submission_filename)
                
                # --- MLflow Log Submission File ---
                mlflow.log_artifact(submission_filename)
                # --- End MLflow Log Submission File ---
            else:
                print("Skipping test prediction and submission due to missing test data or class information.")
        
        print(f"--- Finished Experiment for Model: {model_name} ---")

    print("\nAll experiments complete! Run 'mlflow ui' in your terminal to view results.")


Using CPU
Train Data Directory: crop pictures\train ✓
Val Data Directory: crop pictures\val ✓
Test Data Directory: crop pictures\test ✓
Found 4 classes: ['Blight', 'Common_Rust', 'Gray_Leaf_Spot', 'Healthy']

--- Starting Experiment for Model: VGG16 ---
Created medium augmented transforms (with enhancements).
Created base transforms for image size: (224, 224). Deblurring: wiener, Noise Reduction: bilateral
Training dataset created: 2930 samples
Validation dataset created: 632 samples
Found 626 test images in crop pictures\test
Test submission dataset created: 626 samples
Train DataLoader created with 92 batches.
Val DataLoader created with 20 batches.
Test_submission DataLoader created with 20 batches.
Initialized VGG16 model with pretrained weights.
Model architecture (VGG16):
VGG(
  (features): Sequential(
    (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU(inplace=True)
    (2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
   

Epoch 1/10 Validation: 100%|██████████| 20/20 [02:57<00:00,  8.86s/it]


Epoch 1/10: Train Loss: 1.4541, Train Acc: 0.2904 | Val Loss: 1.3515, Val Acc: 0.3117


Epoch 2/10 Training:  45%|████▍     | 41/92 [12:50:26<15:58:21, 1127.47s/it]  


🏃 View run VGG16_Pretrained at: http://localhost:5000/#/experiments/854895454183404100/runs/750b60faa1544da18a9831d19ce78ceb
🧪 View experiment at: http://localhost:5000/#/experiments/854895454183404100


KeyboardInterrupt: 